# Лабораторная работа №1

## Бинарная сегментация с вероятностной постобработкой


### Цель

Построить воспроизводимый экспериментальный конвейер бинарной сегментации и исследовать, в каких условиях DenseCRF улучшает границы предсказания, а в каких приводит к потере объектов или усилению ошибок базовой модели.

Результатом работы является не отдельная улучшенная маска, а обоснованный вывод о применимости вероятностной постобработки к заданному набору данных.

## 1. Что используется в работе

В открытом режиме репозиторий предоставляет:

- процедурный smoke-набор изображений и бинарных масок;
- CPU-модуль `lab_segmentation.py` с детерминированным baseline;
- вероятностную карту foreground-класса для каждого изображения;
- базовый код расчёта IoU и Dice;
- открытое окружение; `pydensecrf` подключается только для полной CRF-серии.

Smoke-профиль проверяет контракт и не является оценкой качества модели. В полном institutional-профиле преподаватель может предоставить данные и модель с тем же интерфейсом.

Перед smoke-запуском из корня репозитория создайте данные командой `uv run python scripts/generate_smoke_data.py --output block4_up_to_date_CV/methodical-guidelines/students/data`. Каталог `data/` не коммитится.

Студент не обучает сегментационную модель. Основная техническая задача — построить конвейер постобработки, организовать контролируемый эксперимент и оценить компромисс между качеством маски, качеством границ и вычислительными затратами.

Ожидаемая структура проекта:

```text
lab2/
├── data/
│   ├── train/{images,masks}/
│   ├── validation/{images,masks}/
│   └── test/{images,masks}/
├── outputs/
│   ├── baseline/
│   ├── crf/
│   ├── figures/
│   └── runs.jsonl
├── lab_segmentation.py
└── lab2.ipynb
```

## 2. Краткая теоретическая справка

### 2.1. Вероятностный выход сегментационной модели

Для бинарной сегментации модель формирует логиты или вероятности принадлежности каждого пикселя объекту. Бинарная маска получается пороговой обработкой.

Пороговая маска теряет информацию об уверенности модели. Для DenseCRF используется именно вероятностная карта, а не уже бинаризованный результат.

### 2.2. DenseCRF

DenseCRF рассматривает итоговую разметку как конфигурацию случайных переменных и минимизирует энергию

- **Unary-потенциал** задаётся вероятностями базовой модели.
- **Spatial-потенциал** поощряет одинаковые метки у близких пикселей.
- **Bilateral-потенциал** учитывает одновременно расстояние и сходство цвета.

DenseCRF не добавляет семантического знания. Он перераспределяет метки с опорой на уверенность модели и локальную структуру изображения. Поэтому он может уточнить границы, но не способен надёжно восстановить объект, который модель не обнаружила.

### 2.3. Что означает улучшение

Рост IoU или Dice не гарантирует улучшения границ. Улучшение границ не гарантирует сохранения мелких объектов. Поэтому в работе используются три группы показателей:

- качество области: IoU и Dice;
- качество границы: Boundary F-score;
- стоимость: время постобработки.

Оценка должна проводиться по серии изображений, а не по одному удачному примеру.

## 3. Задачи

1. Проанализировать предоставленный датасет и сформировать опорный набор из **12 изображений**.
2. Получить baseline-предсказания и вероятностные карты.
3. Реализовать DenseCRF-постобработку.
4. Сравнить три обязательных режима:
   - без CRF;
   - CRF только с spatial-потенциалом;
   - CRF со spatial- и bilateral-потенциалами.
5. Исследовать один дополнительный фактор:
   - число итераций;
   - вес spatial-потенциала;
   - вес bilateral-потенциала;
   - масштаб пространственного ядра;
   - чувствительность к калибровке вероятностной карты.
6. Сравнить качество областей, границ и время обработки.
7. Проанализировать случаи улучшения и ухудшения результата.
8. Сформулировать выводы в пределах проведённого эксперимента.

## 4. Подготовка данных и опорного набора

Опорный набор должен включать 12 изображений и покрывать:

- крупные и мелкие объекты;
- простые и сложные границы;
- высококонтрастные и слабоконтрастные области;
- успешные и ошибочные baseline-предсказания;
- минимум три изображения с разрывами или пропусками объекта;
- минимум три изображения с ложноположительными областями.

Почти идентичные кадры одной последовательности не должны занимать более двух позиций.

Заполните таблицу покрытия до запуска экспериментов.

In [ ]:
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

DATA_ROOT = Path("data")
DATA_PROFILE = "smoke"  # smoke или full
DATA_SPLIT = "test"
IMAGE_DIR = DATA_ROOT / DATA_SPLIT / "images" if DATA_PROFILE == "smoke" else DATA_ROOT / "images"
MASK_DIR = DATA_ROOT / DATA_SPLIT / "masks" if DATA_PROFILE == "smoke" else DATA_ROOT / "masks"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Images:", IMAGE_DIR.resolve())
print("Masks:", MASK_DIR.resolve())
print("Outputs:", OUTPUT_DIR.resolve())

In [ ]:
# TODO: сформируйте таблицу анализа датасета.
# Добавьте строки для выбранных 12 изображений.

support_set = pd.DataFrame(columns=[
    "image_id",
    "image_path",
    "mask_path",
    "object_scale",          # small / medium / large
    "boundary_complexity",   # low / medium / high
    "contrast",              # low / medium / high
    "baseline_case",         # expected_good / expected_error
    "selection_reason",
])

support_set

**Контрольная точка 1**

Перед продолжением должны быть готовы:

- таблица опорного набора;
- визуальный обзор выбранных изображений и масок;
- краткое обоснование покрытия факторов;
- проверка отсутствия почти идентичных кадров.

In [ ]:
# TODO: реализуйте визуальный обзор выбранных изображений и масок.
# Требование: единый контактный лист с подписями image_id.

def show_support_set(table: pd.DataFrame, max_items: int = 15) -> None:
    raise NotImplementedError


## 5. Базовая модель и вероятностные карты

Модуль базовой модели предоставляется преподавателем. Он должен возвращать:

- исходный размер изображения;
- вероятностную карту foreground-класса `float32` в диапазоне `[0, 1]`;
- бинарную baseline-маску;
- время инференса.

### Контракт

```python
result = predictor.predict(image_path)

result.probability   # np.ndarray [H, W], float32
result.mask          # np.ndarray [H, W], uint8/bool
result.elapsed_seconds
result.metadata
```

Открытая реализация `open-smoke` входит в репозиторий. В полном режиме модуль можно заменить без изменения остального ноутбука.

In [ ]:
from lab_segmentation import SegmentationPredictor

predictor = SegmentationPredictor(model_variant="open-smoke")
print("Provider:", predictor.model_variant)

In [ ]:
# TODO: после подключения модуля выполните один демонстрационный вызов
# и проверьте размеры, диапазон вероятностей и типы данных.

def validate_prediction(image: np.ndarray,
                        probability: np.ndarray,
                        mask: np.ndarray) -> None:
    assert probability.ndim == 2
    assert mask.ndim == 2
    assert probability.shape == mask.shape
    assert probability.min() >= 0.0
    assert probability.max() <= 1.0

# result = predictor.predict(support_set.iloc[0]["image_path"])
# validate_prediction(...)

print("Добавьте демонстрационный вызов после подключения predictor.")

## 6. Реализация экспериментального конвейера

### 6.1. Метрики областей

Используйте единое определение foreground-класса для всех метрик. Обрабатывайте пустые маски явно, чтобы избежать деления на ноль.

In [ ]:
def binary_iou(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = y_true.astype(bool)
    y_pred = y_pred.astype(bool)
    intersection = np.logical_and(y_true, y_pred).sum()
    union = np.logical_or(y_true, y_pred).sum()
    return 1.0 if union == 0 else float(intersection / union)


def binary_dice(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = y_true.astype(bool)
    y_pred = y_pred.astype(bool)
    intersection = np.logical_and(y_true, y_pred).sum()
    denominator = y_true.sum() + y_pred.sum()
    return 1.0 if denominator == 0 else float(2 * intersection / denominator)

### 6.2. Метрика границ

Boundary F-score сопоставляет границы предсказания и эталона с допустимым пространственным отклонением. Радиус допуска должен быть одинаковым во всех сериях и указан в журнале эксперимента.

Ниже задан интерфейс. Реализацию выполняет студент.

In [ ]:
def boundary_f1(y_true: np.ndarray,
                y_pred: np.ndarray,
                tolerance_px: int = 2) -> float:
    """Вычислить F1 для границ бинарных масок.

    Требования:
    1. Извлечь границы обеих масок.
    2. Учесть совпадение в пределах tolerance_px.
    3. Корректно обработать пустые границы.
    """
    pass

### 6.3. DenseCRF

Функция должна принимать исходное RGB-изображение и вероятностную карту foreground-класса. Unary-потенциалы формируются из вероятностей двух классов: background и foreground.

Не бинаризуйте вероятность до построения unary-потенциалов.

In [ ]:
def apply_dense_crf(
    image_rgb: np.ndarray,
    foreground_probability: np.ndarray,
    *,
    iterations: int,
    spatial_weight: float,
    spatial_sxy: float,
    bilateral_weight: float,
    bilateral_sxy: float,
    bilateral_srgb: float,
    use_bilateral: bool,
) -> np.ndarray:
    """Вернуть бинарную маску после DenseCRF.

    Ожидаемая последовательность:
    1. Проверить и ограничить вероятности.
    2. Сформировать распределение [background, foreground].
    3. Создать unary-потенциалы.
    4. Добавить spatial pairwise term.
    5. При необходимости добавить bilateral pairwise term.
    6. Выполнить inference.
    7. Вернуть argmax-маску foreground-класса.
    """
    pass

### 6.4. Конфигурация эксперимента

Каждый запуск должен определяться машиночитаемой конфигурацией. Изменение нескольких факторов одновременно допускается только в заранее спроектированной факторной серии.

In [ ]:
from dataclasses import dataclass, asdict

@dataclass(frozen=True)
class CRFConfig:
    name: str
    iterations: int = 5
    spatial_weight: float = 3.0
    spatial_sxy: float = 3.0
    bilateral_weight: float = 5.0
    bilateral_sxy: float = 50.0
    bilateral_srgb: float = 10.0
    use_bilateral: bool = True


required_configs = [
    {"name": "baseline", "use_crf": False},
    {
        "name": "spatial_only",
        "use_crf": True,
        "config": CRFConfig(
            name="spatial_only",
            use_bilateral=False,
            bilateral_weight=0.0,
        ),
    },
    {
        "name": "spatial_bilateral",
        "use_crf": True,
        "config": CRFConfig(name="spatial_bilateral"),
    },
]

required_configs

Значения в шаблоне являются **стартовой конфигурацией**, а не доказанно оптимальными параметрами. Они нужны для проверки работоспособности конвейера. Дополнительное исследование должно опираться на сформулированную гипотезу.

In [ ]:
# TODO: сформулируйте гипотезу и создайте дополнительную серию конфигураций.
#
# Пример формата:
# research_question = "Как ... влияет на ... при неизменных ...?"
# hypothesis = "..."
# extra_configs = [...]

research_question = ""
hypothesis = ""
extra_configs = []

### 6.5. Пакетный запуск и журналирование

Реализуйте функцию, которая:

- проходит по опорному набору и конфигурациям;
- получает baseline-предсказание один раз на изображение;
- запускает CRF для требуемых конфигураций;
- сохраняет маски и конфигурации;
- измеряет время;
- рассчитывает метрики;
- записывает результат каждой попытки в `outputs/runs.jsonl`;
- не повторяет уже завершённые конфигурации;
- сохраняет статус и текст ошибки при неуспешном запуске.

In [ ]:
RUNS_PATH = OUTPUT_DIR / "runs.jsonl"

def run_experiment(
    support_table: pd.DataFrame,
    predictor,
    configurations: list,
    runs_path: Path = RUNS_PATH,
) -> pd.DataFrame:
    """Выполнить воспроизводимую серию экспериментов."""
    pass

**Контрольная точка 2**

Конвейер считается готовым, если:

- один baseline-инференс не повторяется для каждой CRF-конфигурации;
- все параметры сохраняются вместе с результатом;
- повторный запуск пропускает завершённые эксперименты;
- ошибки одного изображения не останавливают всю серию;
- по журналу можно восстановить каждую маску и её конфигурацию.

## 7. Проверка, оценка и представление результатов

Обязательное сравнение проводится на одном и том же опорном наборе:

1. baseline;
2. spatial-only CRF;
3. spatial + bilateral CRF.

Для дополнительного исследования остальные существенные параметры должны оставаться неизменными.

In [ ]:
# TODO: запустите обязательные и дополнительные конфигурации.
# all_configs = required_configs + extra_configs
# results = run_experiment(support_set, predictor, all_configs)
# results.head()

results = pd.DataFrame()
results

In [ ]:
# TODO: сформируйте сводную таблицу.
# Минимальные агрегаты:
# - mean, median, std для IoU, Dice, Boundary F1;
# - mean и p95 для времени;
# - доля изображений с улучшением и ухудшением каждой метрики.

def summarize_results(results: pd.DataFrame) -> pd.DataFrame:
    raise NotImplementedError

# summary = summarize_results(results)
# summary

In [ ]:
# TODO: постройте контактные листы:
# image / ground truth / baseline / spatial-only / spatial+bilateral.
# Используйте одинаковый порядок изображений и подписи с изменением метрик.

def make_comparison_sheet(results: pd.DataFrame,
                          image_ids: list[str],
                          output_path: Path) -> None:
    raise NotImplementedError

In [ ]:
# TODO: постройте не менее двух графиков:
# 1. качество областей и границ по режимам;
# 2. время постобработки по режимам или исследуемому фактору.
#
# Не смешивайте несопоставимые конфигурации на одном графике без пояснения.

### Анализ характерных случаев

Выберите:

- не менее трёх изображений, где CRF улучшил результат;
- не менее трёх изображений, где CRF ухудшил результат.

Для каждого случая укажите:

- изменение IoU, Dice и Boundary F-score;
- что изменилось визуально;
- какая часть модели CRF могла вызвать эффект;
- связан ли эффект с цветовой границей, размером объекта, ошибкой вероятностной карты или выбранными параметрами.

## 8. Анализ и сдача

В итоговом анализе разделите:

1. **наблюдения** — непосредственно измеренные изменения.
2. **интерпретацию** — предполагаемая причина этих изменений.
3. **границы вывода** — условия, которые реально были проверены.

Ответьте на исследовательский вопрос и укажите:

- улучшает ли CRF среднее качество областей;
- улучшает ли CRF качество границ;
- какие классы ошибок он исправляет;
- какие ошибки усиливает;
- оправдана ли вычислительная стоимость;
- можно ли рекомендовать выбранную конфигурацию для всего датасета.

### Обязательные артефакты

1. Заполненная таблица опорного набора.
2. Визуальный обзор 12–15 изображений.
3. Baseline-маски и вероятностные карты.
4. Реализация Boundary F-score.
5. Реализация DenseCRF.
6. Воспроизводимый пакетный конвейер.
7. `runs.jsonl` со всеми попытками.
8. Сравнение трёх обязательных режимов.
9. Одно дополнительное исследование.
10. Сводная таблица метрик и времени.
11. Не менее двух графиков.
12. Контактные листы.
13. Анализ трёх успешных и трёх неуспешных случаев.
14. Итоговые выводы.

### Критерии оценивания

- Анализ датасета и выбор опорного набора
- Корректность реализации Boundary F-score и DenseCRF
- Реализация воспроизводимого конвейера
- Обязательные и дополнительное исследования
- Оценка, визуализация и анализ результатов
- Воспроизводимость и полнота материалов
